# 1. Objective
Prepare the cleaned pothole records for the classification models (Random Forest, XGBoost, LightGBM, Gradient Boosting, SVM).

# 2. Dataset Used
Load the cleaned dataset and inspect it.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../dataset/processed/pothole_dataset_cleaned.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Records:", df.duplicated().sum())
print("\nTarget Distribution:\n", df['class'].value_counts())


Shape: (1886, 14)
Columns: ['filename', 'img_width', 'img_height', 'class', 'xmin', 'ymin', 'xmax', 'ymax', 'bbox_width', 'bbox_height', 'bbox_area', 'rel_bbox_width', 'rel_bbox_height', 'rel_bbox_area']

Missing Values:
 filename           0
img_width          0
img_height         0
class              0
xmin               0
ymin               0
xmax               0
ymax               0
bbox_width         0
bbox_height        0
bbox_area          0
rel_bbox_width     0
rel_bbox_height    0
rel_bbox_area      0
dtype: int64

Duplicate Records: 0

Target Distribution:
 class
medium_pothole    868
major_pothole     600
minor_pothole     418
Name: count, dtype: int64


# 3. Target Definition
The target is the `class` column, containing `minor_pothole`, `medium_pothole`, and `major_pothole`.

# 4. Feature Selection
Identifiers like `filename` are excluded as predictors but kept for grouping logic.

# 5. Feature Engineering
Creating additional mathematically valid features:
- `bbox_aspect_ratio` = bbox_width / bbox_height
- `log_bbox_area` = log1p(bbox_area)
- Relative dimensions using actual image dimensions.

In [2]:
# bbox_aspect_ratio = bbox_width / bbox_height
df['bbox_aspect_ratio'] = df['bbox_width'] / df['bbox_height'].replace(0, np.nan)
df['bbox_aspect_ratio'] = df['bbox_aspect_ratio'].fillna(0)

# log_bbox_area = log1p(bbox_area)
df['log_bbox_area'] = np.log1p(df['bbox_area'])

# Normalized width/height
if 'rel_bbox_width' not in df.columns:
    df['rel_bbox_width'] = df['bbox_width'] / df['img_width']
if 'rel_bbox_height' not in df.columns:
    df['rel_bbox_height'] = df['bbox_height'] / df['img_height']

print("Engineered Features Added.")
df[['bbox_aspect_ratio', 'log_bbox_area', 'rel_bbox_width', 'rel_bbox_height']].head()


Engineered Features Added.


,bbox_aspect_ratio,log_bbox_area,rel_bbox_width,rel_bbox_height
0,2.260870,10.670280,0.433333,0.191667
1,1.127660,10.017709,0.220833,0.195833
2,1.715517,10.046938,0.276389,0.161111
3,1.286822,9.971847,0.230556,0.179167
4,1.670213,9.599608,0.218056,0.130556


# 6. Leakage Considerations
Bounding-box dimensions are extracted from the manual object annotations and show a strong association with the annotated severity class. Therefore, the resulting classification performance may partly reflect the relationship between annotated object size and the severity labels.

# 7. Target Encoding
Encode classes numerically while preserving original labels for mapping back later.

In [3]:
le = LabelEncoder()
df['target'] = le.fit_transform(df['class'])

class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Class Mapping:", class_mapping)

os.makedirs('../models', exist_ok=True)
joblib.dump(le, '../models/label_encoder.pkl')


Class Mapping: {'major_pothole': 0, 'medium_pothole': 1, 'minor_pothole': 2}


['../models/label_encoder.pkl']

# 8. Train/Validation/Test Split
Using a grouped split based on `filename` (image ID) to prevent records from the same image appearing in different splits, avoiding data leakage.

In [4]:
# We use GroupShuffleSplit to ensure records from the same image stay in the same split.
features = ['bbox_width', 'bbox_height', 'bbox_area', 'rel_bbox_width', 'rel_bbox_height', 'rel_bbox_area', 'bbox_aspect_ratio', 'log_bbox_area']
X = df[features]
y = df['target']
groups = df['filename']

# Train (70%), Temp (30%)
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss1.split(X, y, groups=groups))

X_train, y_train, groups_train = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
X_temp, y_temp, groups_temp = X.iloc[temp_idx], y.iloc[temp_idx], groups.iloc[temp_idx]

# Valid (15%), Test (15%)
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
valid_idx, test_idx = next(gss2.split(X_temp, y_temp, groups=groups_temp))

X_valid, y_valid = X_temp.iloc[valid_idx], y_temp.iloc[valid_idx]
X_test, y_test = X_temp.iloc[test_idx], y_temp.iloc[test_idx]

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)


X_train shape: (1314, 8)
X_valid shape: (303, 8)
X_test shape: (269, 8)
y_train shape: (1314,)


# 9. Class Distribution
Displaying class distribution across all splits. If imbalanced, it will be evaluated during modeling.

In [5]:
print("Full Dataset Distribution:\n", df['target'].value_counts(normalize=True))
print("\nTraining Set Distribution:\n", y_train.value_counts(normalize=True))
print("\nValidation Set Distribution:\n", y_valid.value_counts(normalize=True))
print("\nTest Set Distribution:\n", y_test.value_counts(normalize=True))


Full Dataset Distribution:
 target
1    0.460233
0    0.318134
2    0.221633
Name: proportion, dtype: float64

Training Set Distribution:
 target
1    0.469559
0    0.328767
2    0.201674
Name: proportion, dtype: float64

Validation Set Distribution:
 target
1    0.465347
0    0.277228
2    0.257426
Name: proportion, dtype: float64

Test Set Distribution:
 target
1    0.408922
0    0.312268
2    0.278810
Name: proportion, dtype: float64


# 10. Preprocessing Strategy
Tree-based models do not require scaling. SVM requires standard scaling. The scaler is fitted only on the training data.

In [6]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_valid_scaled = pd.DataFrame(scaler.transform(X_valid), columns=X_valid.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

joblib.dump(scaler, '../models/scaler.pkl')

X_train.to_csv('../dataset/processed/X_train.csv', index=False)
X_valid.to_csv('../dataset/processed/X_valid.csv', index=False)
X_test.to_csv('../dataset/processed/X_test.csv', index=False)
y_train.to_csv('../dataset/processed/y_train.csv', index=False)
y_valid.to_csv('../dataset/processed/y_valid.csv', index=False)
y_test.to_csv('../dataset/processed/y_test.csv', index=False)

X_train_scaled.to_csv('../dataset/processed/X_train_scaled.csv', index=False)
X_valid_scaled.to_csv('../dataset/processed/X_valid_scaled.csv', index=False)
X_test_scaled.to_csv('../dataset/processed/X_test_scaled.csv', index=False)

print("Datasets and Preprocessing artifacts saved.")


Datasets and Preprocessing artifacts saved.


# 11. Dataset Limitations
The supplied dataset is primarily an image/object-detection dataset. The current tabular severity-classification pipeline uses features extracted from Pascal VOC bounding-box annotations.
The classification model in this phase is a tabular classifier operating on features extracted from the object annotations, rather than a deep-learning image classifier.

The dataset does not provide:
* road-condition measurements
* traffic exposure
* environmental variables
* geographic road-segment data
* maintenance-cost observations

Therefore, those variables cannot be used as actual predictive features unless they are obtained from another legitimate source.

